In [1]:
import torch

from peft import PeftModel
from transformers import RobertaTokenizer, RobertaModel, AutoTokenizer, AutoModelForCausalLM

In [2]:
batch_size = 1
sequence_length = 512
hidden_size = 768
block_id = 1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# 初始化模型参数
test_sentence = 'Hello World! This is a test!'
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
input = tokenizer(test_sentence, return_tensors='pt').to(device)
print(input['input_ids'].shape)

torch.Size([1, 10])


/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
# 加载预训练模型
public_model = AutoModelForCausalLM.from_pretrained('roberta-base').to(device)
print(public_model)
# 执行推理
public_output = public_model(**input, output_hidden_states=True)

If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`


RobertaForCausalLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [5]:
print(public_output.hidden_states[0].shape)
print(len(public_output.hidden_states))

torch.Size([1, 10, 768])
13


In [6]:
# 加载微调的模型
base_model = AutoModelForCausalLM.from_pretrained("roberta-base")
private_model = PeftModel.from_pretrained(base_model, "Tech-oriented/Roberta_peft_model").to(device)
private_model = private_model.merge_and_unload()
# private_model = private_model.unload()
del base_model
print(private_model)

# 执行推理
private_output = private_model(**input, output_hidden_states=True)

If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`


RobertaForCausalLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [7]:
print(private_output.hidden_states[0].shape)
print(len(private_output.hidden_states))

torch.Size([1, 10, 768])
13


In [8]:
# 查看中间结果相似度
public_hidden_state = public_output.hidden_states[block_id]
private_hidden_state = private_output.hidden_states[block_id]

print(torch.cosine_similarity(public_hidden_state, private_hidden_state, dim=2))

tensor([[1.0000, 0.9998, 0.9998, 0.9999, 0.9999, 0.9999, 0.9999, 0.9999, 0.9999,
         0.9999]], device='cuda:0', grad_fn=<SumBackward1>)


In [9]:
# 狠狠地扰动私有结果
def generate_random_permutation_matrix(n):
    # 创建一个单位矩阵
    identity_matrix = torch.eye(n)

    # 生成一个随机排列的索引
    permuted_indices = torch.randperm(n)

    # 打乱单位矩阵的行/列顺序
    permutation_matrix = identity_matrix[permuted_indices]

    return permutation_matrix.to(device)

permutation_key = generate_random_permutation_matrix(hidden_size)
print(permutation_key @ torch.inverse(permutation_key)) # 必须仅输出单位阵
print(permutation_key @ permutation_key.T) # 必须仅输出单位阵
private_hidden_state_Cipher = private_hidden_state @ permutation_key
print(torch.cosine_similarity(public_hidden_state, private_hidden_state_Cipher, dim=2))

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]], device='cuda:0')
tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]], device='cuda:0')
tensor([[ 0.0023,  0.0180,  0.0290,  0.0191, -0.0356, -0.0044, -0.0129, -0.0407,
          0.0214,  0.0255]], device='cuda:0', grad_fn=<SumBackward1>)


In [10]:
# 将行列转置
public_hidden_state_T = public_hidden_state.squeeze().T
private_hidden_state_Cipher_T = private_hidden_state_Cipher.squeeze().T
print(public_hidden_state_T.shape)

hidden_state_length = public_hidden_state_T.shape[0]
similarity_matrix = torch.ones(hidden_state_length, hidden_state_length).to(device)

import tqdm

# 计算相似度矩阵，用于Jonker-Volgenant匹配
for i in tqdm.tqdm(range(hidden_state_length)):
    for j in range(hidden_state_length):
        similarity_matrix[i][j] = 10 - 10*torch.cosine_similarity(public_hidden_state_T[i], private_hidden_state_Cipher_T[j], dim=0)

print(similarity_matrix)

torch.Size([768, 10])


100%|██████████| 768/768 [01:14<00:00, 10.31it/s]

tensor([[ 8.0443,  7.4353,  6.5238,  ...,  9.9167, 12.8213, 14.5029],
        [14.6946, 15.3578,  4.7675,  ..., 14.2599, 16.7608, 14.7998],
        [ 3.9676,  4.3922, 15.8909,  ...,  4.5753,  1.4248,  4.5993],
        ...,
        [11.7376, 12.8141, 14.2205,  ..., 12.6809,  7.3307, 14.2535],
        [12.3976,  7.3411,  7.2806,  ...,  6.4521,  8.2270,  4.9570],
        [ 8.3829, 11.8031,  3.7496,  ...,  9.3124,  9.2225,  6.2029]],
       device='cuda:0', grad_fn=<CopySlices>)


In [11]:
from scipy.optimize import linear_sum_assignment
# Jonker-Volgenant匹配
row_ind, col_ind = linear_sum_assignment(similarity_matrix.cpu().detach().numpy())

# 从匹配结果row_ind, col_ind中提取最优的排列矩阵，近似恢复 permutation_key.T
recovered_permutation_matrix = torch.zeros(hidden_state_length, hidden_state_length).to(device)
recovered_permutation_matrix[row_ind, col_ind] = 1
print(torch.sum(recovered_permutation_matrix))
recovered_permutation_matrix = recovered_permutation_matrix.T

tensor(768., device='cuda:0')


In [ ]:
print('Testing the unsealed KEY---------------------------------------------------------\n')
print("Testing the Cosine_similarity between recovered X and real X (should all be 1):")
cos_between_recovered_x_and_real_x = torch.cosine_similarity(public_hidden_state, private_hidden_state_Cipher @ 
                                                             recovered_permutation_matrix, dim=2)
print(cos_between_recovered_x_and_real_x)
# 误差应当定义大一些
test_result_1 = torch.allclose(cos_between_recovered_x_and_real_x, torch.ones_like(cos_between_recovered_x_and_real_x), rtol=0.1, atol=0.1)
print(test_result_1, '\n')

print("Testing the property of unseal KEY (should be unit matrix):")
print(permutation_key @ recovered_permutation_matrix)
test_result_2 = torch.allclose(permutation_key @ recovered_permutation_matrix, torch.eye(recovered_permutation_matrix.shape[0], device=device), atol=0.1, rtol=0.1)
print(test_result_2, '\n')

if test_result_1 and test_result_2:
    print("Attack success!")
else:
    print("Attack failed!")

Testing the unsealed KEY---------------------------------------------------------

Testing the Cosine_similarity between recovered X and real X (should all be 1):
tensor([[1.0000, 0.9998, 0.9998, 0.9999, 0.9999, 0.9999, 0.9999, 0.9999, 0.9999,
         0.9999]], device='cuda:0', grad_fn=<SumBackward1>)
True 

Testing the property of unseal KEY (should be unit matrix):
tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]], device='cuda:0')
True 

Attack success!
